# Teste e depuração

Um bloco por script. **Cada bloco é autocontido**: rode o bloco de partida uma vez e
depois qualquer bloco isolado, na ordem que quiser.

Os blocos chamam as funções do `pipeline_core` em vez de remontar o comando — assim o CLI
de cada script continua existindo num lugar só, e um bloco daqui nunca fica desatualizado
em relação ao pipeline.

Tudo roda com `NEGSEC_SEM_EMAIL=1`.

In [ ]:
# Bloco de partida — rode uma vez por sessão.
# Ele é o que torna todos os blocos abaixo AUTOCONTIDOS: cada um pode ser rodado
# sozinho, sem depender dos anteriores.
import os, sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "rotinas" else Path.cwd()
sys.path.insert(0, str(RAIZ / "codigos" / "helpers"))
os.chdir(RAIZ)

# Em teste e em lote, sempre sem email: evita spam e a instância COM órfã do Outlook.
os.environ["NEGSEC_SEM_EMAIL"] = "1"

import dados as D
import pipeline_core as pc
print("raiz:", RAIZ)
print("pregões na base:", len(D.Datas("NegociosProcessados")))

## Datas de referência

A maioria dos scripts trabalha sobre uma liquidação `X` e a anterior `X-1u`.

In [ ]:
X    = pc.UltimosNDiasUteis(1)[0]      # último pregão
Xant = pc.DiaUtilAnterior(X)
print("X =", X, " X-1u =", Xant)

---
## 1 · Boletim B3 — os negócios

Única fonte de negócio do projeto.

In [ ]:
pc.Boletim(Xant, X)

## 2 · Cadastro pela B3 — fonte primária

In [ ]:
pc.BondDetails(Xant, X)

## 3 · Taxas indicativas — debêntures

In [ ]:
pc.AnbimaDeb(Xant)
pc.AnbimaDeb(X)

## 4 · Taxas indicativas — CRI e CRA

Playwright. O portal guarda ~5 pregões.

In [ ]:
pc.AnbimaCriCra(X)

## 5 · Planilha FI Analytics

Playwright + login. É o script mais frágil — falha aqui não derruba o pipeline.

In [ ]:
pc.FiAnalytics()

## 6 · Cadastro pela Anbima Data

O passo mais lento. Use `limit` para smoke test.

In [ ]:
pc.AnbimaData(Xant, X, limit=20)

## 7 · Curva NTN-B

In [ ]:
pc.Ntnb(Xant, X)

## 8 · Curva DI

Dois destinos num download: `MtmAnbima` (DI1) e `CurvaDi`.

In [ ]:
pc.CurvaDi(X)

## 9–11 · Insumos da calculadora

Não dependem de `X` — uma passada por ciclo.

In [ ]:
pc.IpcaIbge()
pc.IpcaProjetado()   # rodar antes das 17h30
pc.DiBcb()

## 12 · Outstanding (Bloomberg)

**Só no banco.** Sem terminal, fica vazio e a Visão Anbima usa proxy.

In [ ]:
pc.Outstanding(Xant, X)

## 13 · O gate

Decide em quais ativos a calc local pode precificar. Roda a base inteira; use `negociadosDias` para encurtar.

In [ ]:
pc.ValidarCalcB3(negociadosDias=5)

## 14 · Taxa por negócio

`limit` prioriza os que precisam de API — é o smoke test certo.

In [ ]:
pc.CalcTaxa(X, limit=40)

## 15 · Filtro de duplicados

In [ ]:
pc.Filtrar(X)

## 16 · PU par

Idempotente por (ativo, data): reprocessar custa zero chamada.

In [ ]:
pc.PuPar(X)

## 17 · Spread das indicativas

Roda em `X-1u` — é a data que o relatório exibe.

In [ ]:
pc.SpreadAnbima(Xant)

## 18 · Match de referência

Sem data: global e idempotente.

In [ ]:
pc.MatchRef()

## 19 · Spread dos negócios

In [ ]:
pc.SpreadOver(X)

## 20 · Relatório

Toda a base. Confira o número de aceitação.

In [ ]:
pc.Relatorio()

---
## Ferramentas

Não são passos do pipeline.

In [ ]:
# contrato da camada de dados (não encosta na base real)
pc.RodarPasso("conferir_dados")

In [ ]:
# contrato da ponte com a calculadora
pc.RodarPasso("conferir_calc")

In [ ]:
# trava de segurança — rode ANTES de qualquer push
pc.RodarPasso("check_no_secrets")